<a href="https://colab.research.google.com/github/cfmiila/Agente_Triagem_Medica.ipynb/blob/main/Base_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base para Agent LangChain

## Setup e Libs

In [1]:
!pip install -qU langchain

In [2]:
!pip install -qU langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 10.9 MB/s eta 0:00:00


In [3]:
!pip install -qU langchain-tavily

In [4]:
!pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.5 MB/s eta 0:00:00


In [5]:
import os
from google.colab import userdata

# 1. Puxa as chaves cadastradas na aba lateral "Secrets" do Colab
google_key = userdata.get('GEMINI_API_KEY')

# 2. Configura as chaves como variáveis de ambiente para o LangChain encontrar
os.environ["GEMINI_API_KEY"] = google_key

print("✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!")

✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!


In [6]:
tavily_api_key = userdata.get('TAVILY_API_KEY')
os.environ["TAVILY_API_KEY"] = tavily_api_key
print("✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!")


✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!


In [8]:
from langchain_tavily import TavilySearch
tavily_web_search_tool = TavilySearch(
    max_results=3,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [9]:
query= "Qual é o líder do campeonato brasileiro de 2026?"
# Executa a busca
resultados = tavily_web_search_tool.invoke({"query": query})['results']

# Exibe os resultados
for i, res in enumerate(resultados, 1):
    print(f"--- Resultado {i} ---")
    print(f"URL: {res['url']}")
    print(f"Conteúdo: {res['content']}\n")

--- Resultado 1 ---
URL: https://www.instagram.com/reel/DVz1o-zCq4d/
Conteúdo: O São Paulo é o novo líder do Campeonato Brasileiro de 2026, após vitória de 2 a 0 contra a Chapecoense, na estreia do técnico Roger Machado. O

--- Resultado 2 ---
URL: https://www.estadao.com.br/esportes/futebol/campeonatos/brasileirao-serie-a-2026/?srsltid=AfmBOop8ivrWJRuGmSBYVlQS284lFX5f8NpFVo_JpN-EZUo4fcC_YNNP
Conteúdo: [](https://www.estadao.com.br/ "Home Estadão"). [Tabelas](https://www.estadao.com.br/esportes/futebol/campeonatos/ "Tabelas"). ![Image 11](https://arte.estadao.com.br/public/esportes/brasoes/thumbs/50/vitoria.png). ![Image 13](https://arte.estadao.com.br/public/esportes/brasoes/thumbs/50/gremio.png). ![Image 14](https://arte.estadao.com.br/public/esportes/brasoes/thumbs/50/internacional.png). ![Image 15](https://arte.estadao.com.br/public/esportes/brasoes/thumbs/50/santos.png). ![Image 16](https://arte.estadao.com.br/public/esportes/brasoes/thumbs/50/cruzeiro.png). ![Image 17](https://ar

**AGENTE**

In [10]:
system_prompt = "Você é um assistente de buscas na web. Você é prestativo e educado. Use as ferramentas disponíveis para responder as demandas do usuário"

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model= 'gemini-2.5-flash', temperature=0.1)

tools = [tavily_web_search_tool]


In [12]:
from langchain.agents import create_agent
agent = create_agent(model=llm,tools=tools, system_prompt=system_prompt)

In [20]:
agent.invoke({"messages": [("user", "PERGUNTA: quando o homem foi a lua pela primeira e pela última vez?")]})

{'messages': [HumanMessage(content='PERGUNTA: quando o homem foi a lua pela primeira e pela última vez?', additional_kwargs={}, response_metadata={}, id='198150fc-47af-4788-9430-0a6ac266c1ff'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "quando o homem foi a lua pela \\u00faltima vez"}'}, '__gemini_function_call_thought_signatures__': {'aa0ff607-f0b5-4b07-bf36-9fb6958b94e5': 'CrICAQw51scIG7wFzIPluQS4kpS7u5SGiTlyTSIao49g87zXqe7l+MZJC/y/h5LO1bv72Qg/eaILDobmllnvrHpuZqf4woKOXpjxi48aMZUf5eWM0g5n6PHXDVaYDo4SVGt9MYMU/Dh7S5vV6m2jmZHolmbYYxlSU3vXEXICLXYHbHTy5kc4w2432hsU+yz2Umirb7mQBWD4veUzZ+84uxaqUwHCLYfaNrXj+ZJNVz9QSlholQ7AnM+CVjJbVRxRamAvRLU82FsNC6gMseKkJsUNkSKvMYDDDwbFZToup1Q2CfiBUIg7+21NE4fu0fZxdWa16xOA5UEjgXlK77e3ZLH/eF8++oOQ1UouEcqZBndRD0+L53zyX2MMKXR/FJZ7yfhiV7/MBBI2jck57JsGGTPBnimS'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.1)

tools = [tavily_web_search_tool]

{'messages': [HumanMessage(content='PERGUNTA: quando o homem foi a lua pela primeira e pela última vez?', additional_kwargs={}, response_metadata={}, id='fc04882f-3e6b-4699-b8de-7dffe9aad5bf'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "quando o homem foi a lua pela \\u00faltima vez"}'}, '__gemini_function_call_thought_signatures__': {'65a07f62-05e5-40ef-b8ce-da63902c19f3': 'CrICAQw51sfKf4yqK3oaYxIDvtjfkfbeMJQmScZGYLnlFyzEG3aJl3NlMQ8zKGWELhdZyvAwPgY6csx2fu/cxYjLLO81UmD3f/Leyh3hm7EDbjJBcQbZO4Y3l3+oWKYgrxl4H1yLrb0wiaQxcW471Os9U7rbDxHloWpxmycifYDDizKK+YXphvmj2KijRlxHJ3IUgL+gJpdRF4Yk3yzmGjlIdE17ghy1xL24SqM+nOvScUChONqKH+msKNtqnPfvN8LSleN3U7QGYEzM+C5Z/pbdiB72hza3ZJjwtW/rU6ebn43DwvifKeo08Ii1E3kjYEjpOu/Jp+3m85zI3K+QmVOQiuHEmfngNZT2ZHwtTMwdHLa84kErqC2wkAh/H17fGTc88kB4aAo4JQyW/JdtFbrTuBZ5'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id